In [1]:

# ==========================
# 1. Ligação à base de dados DuckDB
# ==========================
import duckdb
from pathlib import Path

DB_PATH = Path(
    r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\00.DB\2026.duckdb"
)

CON = duckdb.connect(str(DB_PATH), read_only=False)


In [2]:

# ==========================
# 1. Apagar tabelas existentes
# ==========================
CON.execute("DROP TABLE IF EXISTS tarifas_mc_2026_pal")
CON.execute("DROP TABLE IF EXISTS tarifas_mc_2026_peso")

# ==========================
# 2. Criar tabela de paletes
# ==========================
colunas_pal = ",\n".join(
    f"PAL_{i} DECIMAL(12,4)"
    for i in range(1, 34)
)

CON.execute(f"""
    CREATE TABLE tarifas_mc_2026_pal (
        CLIENTE VARCHAR,
        DESIGNACAO VARCHAR,
        LOCDES VARCHAR,
        DISTRITO VARCHAR,
        {colunas_pal}
    )
""")

# ==========================
# 3. Criar tabela de peso
# ==========================
CON.execute("""
    CREATE TABLE tarifas_mc_2026_peso (
        CLIENTE VARCHAR,
        DESIGNACAO VARCHAR,
        LOCDES VARCHAR,
        DISTRITO VARCHAR,
        KG_MAX DECIMAL(12,3),
        TARIFA DECIMAL(12,4)
    )
""")

# ==========================
# 4. Confirmar tabelas
# ==========================
df = CON.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_name IN (
        'tarifas_mc_2026_pal',
        'tarifas_mc_2026_peso'
    )
    ORDER BY table_name
""").df()

df


,table_name
0,tarifas_mc_2026_pal
1,tarifas_mc_2026_peso


In [3]:

# ==========================
# 1. Definir pasta
# ==========================
from pathlib import Path
import pandas as pd

PASTA = Path(r"T:\D-Trafico\02.FACTURAÇÃO\2026\09_SETEMBRO\Emitir")

# ==========================
# 2. Listar ficheiros
# ==========================
print("Pasta existe:", PASTA.is_dir())

df = pd.DataFrame([
    {
        "FICHEIRO": f.name,
        "EXTENSAO": f.suffix.lower(),
        "TAMANHO_MB": round(f.stat().st_size / 1024**2, 2)
    }
    for f in PASTA.iterdir()
    if f.is_file()
]).sort_values("FICHEIRO").reset_index(drop=True)

df


Pasta existe: True


,FICHEIRO,EXTENSAO,TAMANHO_MB
0,09. FACTURAÇÃO - NUTRICIA - SETEMBRO 2026_CLIE...,.xls,1.22
1,Auchan Septiembre Facturación.xlsx,.xlsx,0.14
2,Carmonti Septiembre Facturación.xlsx,.xlsx,0.59
3,Confeitaria Carlos Gonçalves Septiembre Factur...,.xlsx,0.55
4,DF Septiembre Facturación.xlsx,.xlsx,0.44
5,FLORA Septiembre Facturación.xlsx,.xlsx,19.21
6,Factura GL_Septiembre.2026 Facturación.xlsx,.xlsx,16.72
7,Factura Lourofood_Septiembre.26 Facturación.xlsx,.xlsx,16.94
8,Factura Pcarnes_Septiembre Facturación.xlsx,.xlsx,21.05
9,Factura RTM_Septiembre.26 Facturación.xlsx,.xlsx,17.52


In [4]:

# ==========================
# 1. Ler ficheiro
# ==========================
import pandas as pd

ficheiro = PASTA / "Carmonti Septiembre Facturación.xlsx"

df = pd.read_excel(
    ficheiro,
    sheet_name="SALDAT027",
    usecols=["CODACT", "LOCDES", "LUGARDESCARGA", "PROV_ENTREGAR"],
    dtype=str
)

# ==========================
# 2. Organizar destinos
# ==========================
df = (
    df.rename(columns={
        "CODACT": "CLIENTE",
        "LUGARDESCARGA": "DESIGNACAO",
        "PROV_ENTREGAR": "DISTRITO"
    })
    .assign(NOME_CLIENTE="CARMONTI")
    .loc[lambda x: x["CLIENTE"].str.strip() == "403"]
    [["CLIENTE", "NOME_CLIENTE", "LOCDES", "DESIGNACAO", "DISTRITO"]]
    .dropna(subset=["LOCDES"])
    .drop_duplicates()
    .sort_values(["DESIGNACAO", "DISTRITO"])
    .reset_index(drop=True)
)

# ==========================
# 3. Visualizar
# ==========================
df


,CLIENTE,NOME_CLIENTE,LOCDES,DESIGNACAO,DISTRITO
0,403,CARMONTI,711369,"A. PIRES LOURENÇO & FILHOS,LDA",Castelo Branco
1,403,CARMONTI,752607,"ALFREDO DA SILVA BARBOSA,LDA (REALSABOR)",Porto
2,403,CARMONTI,700739,ANMANDIO DE JESUS P.PACHECO,Lisboa
3,403,CARMONTI,312280,ANTONIO GASPAR DE SOUSA,MEALHADA
4,403,CARMONTI,322849,CARMONTI-IND.CARNES DE MONTIJO,Setubal
5,403,CARMONTI,313228,CASA QUINTELA,Castelo Branco
6,403,CARMONTI,784507,CENTRALFUMADOS - COMERCIO E INDUSTRIA DE CARNE...,Guarda
7,403,CARMONTI,784507,CENTRALFUMADOS - COMERCIO E INDUSTRIA DE CARNE...,ÍLHAVO
8,403,CARMONTI,579957,CMTIR,Aveiro
9,403,CARMONTI,579957,CMTIR,Braga


In [5]:

# ==========================
# 1. Bibliotecas
# ==========================
import pandas as pd
import re
import unicodedata
from openpyxl import load_workbook

# ==========================
# 2. Configuração
# ==========================
campos = ["CODACT", "LOCDES", "LUGARDESCARGA", "PROV_ENTREGAR"]

registos = []
resumo = []
erros = []

excluir = ("NUTRICIA", "DANONE", "ALPRO", "COB")

# ==========================
# 3. Normalizar textos
# ==========================
def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto or ""))
    return "".join(
        c for c in texto if not unicodedata.combining(c)
    ).strip().upper()

def limpar_valor(valor):
    if valor is None:
        return None
    valor = str(valor).strip()
    if not valor:
        return None
    return re.sub(r"\.0$", "", valor)

def nome_cliente(ficheiro):
    nome = ficheiro.stem
    nome = re.sub(r"(?i)^fatura[_ ]|^factura[_ ]", "", nome)
    nome = re.split(r"(?i)[_ ]septiembre|[_ ]setembro", nome)[0]
    return nome.strip(" _-")

# ==========================
# 4. Ler ficheiros
# ==========================
for ficheiro in sorted(PASTA.glob("*.xlsx")):

    if ficheiro.name.startswith("~$"):
        continue

    if any(x in normalizar(ficheiro.name) for x in excluir):
        continue

    wb = None

    try:
        wb = load_workbook(
            ficheiro,
            read_only=True,
            data_only=True
        )

        folha = next(
            (s for s in wb.sheetnames
             if "SALDAT027" in normalizar(s)),
            None
        )

        if folha is None:
            raise ValueError("Folha SALDAT027 não encontrada")

        ws = wb[folha]
        linhas = ws.iter_rows(values_only=True)

        indices = None

        for numero, linha in enumerate(linhas, start=1):
            if numero > 30:
                break

            cabecalho = [normalizar(v) for v in linha]

            if all(c in cabecalho for c in campos):
                indices = {c: cabecalho.index(c) for c in campos}
                break

        if indices is None:
            raise ValueError("Cabeçalhos obrigatórios não encontrados")

        quantidade = 0

        for linha in linhas:
            valores = {
                c: limpar_valor(linha[i] if i < len(linha) else None)
                for c, i in indices.items()
            }

            if not valores["CODACT"] or not valores["LOCDES"]:
                continue

            registos.append({
                "CLIENTE": valores["CODACT"],
                "NOME_CLIENTE": nome_cliente(ficheiro),
                "LOCDES": valores["LOCDES"],
                "DESIGNACAO": valores["LUGARDESCARGA"],
                "DISTRITO": valores["PROV_ENTREGAR"],
                "FICHEIRO": ficheiro.name
            })

            quantidade += 1

        resumo.append({
            "FICHEIRO": ficheiro.name,
            "REGISTOS_LIDOS": quantidade,
            "ESTADO": "OK"
        })

    except Exception as e:
        erros.append({
            "FICHEIRO": ficheiro.name,
            "ERRO": str(e)
        })

    finally:
        if wb is not None:
            wb.close()

# ==========================
# 5. Consolidar destinos
# ==========================
colunas = [
    "CLIENTE", "NOME_CLIENTE", "LOCDES",
    "DESIGNACAO", "DISTRITO"
]

df_destinos = pd.DataFrame(
    registos,
    columns=colunas + ["FICHEIRO"]
)

df_destinos = (
    df_destinos[colunas]
    .drop_duplicates()
    .sort_values(["CLIENTE", "LOCDES", "DESIGNACAO", "DISTRITO"])
    .reset_index(drop=True)
)

df_resumo = pd.DataFrame(resumo)
df_erros = pd.DataFrame(erros)

# ==========================
# 6. Resultados
# ==========================
print("Ficheiros lidos:", len(df_resumo))
print("Ficheiros com erro:", len(df_erros))
print("Clientes distintos:", df_destinos["CLIENTE"].nunique())
print("Registos únicos:", len(df_destinos))

display(df_resumo)
display(df_erros)
display(df_destinos)


Ficheiros lidos: 24
Ficheiros com erro: 5
Clientes distintos: 24
Registos únicos: 1937


,FICHEIRO,REGISTOS_LIDOS,ESTADO
0,Auchan Septiembre Facturación.xlsx,195,OK
1,Carmonti Septiembre Facturación.xlsx,861,OK
2,Confeitaria Carlos Gonçalves Septiembre Factur...,758,OK
3,DF Septiembre Facturación.xlsx,701,OK
4,Factura GL_Septiembre.2026 Facturación.xlsx,1507,OK
5,Factura Lourofood_Septiembre.26 Facturación.xlsx,1584,OK
6,Factura Pcarnes_Septiembre Facturación.xlsx,8539,OK
7,Factura RTM_Septiembre.26 Facturación.xlsx,2455,OK
8,Fatura Lactaçores_Septiembre.26 Facturación.xlsx,1234,OK
9,FLORA Septiembre Facturación.xlsx,32513,OK


,FICHEIRO,ERRO
0,Factura_Andros Ambiente_Septiembre26_DEF.xlsx,Folha SALDAT027 não encontrada
1,Factura_Andros congelado_Septiembre26_DEF.xlsx,Folha SALDAT027 não encontrada
2,Factura_Andros refrigerado_Septiembre26_DEF.xlsx,Folha SALDAT027 não encontrada
3,Matutano SETEMBRO.xlsx,Folha SALDAT027 não encontrada
4,PLN Bon Natura_Septiembre.26_ok (1).xlsx,Folha SALDAT027 não encontrada


,CLIENTE,NOME_CLIENTE,LOCDES,DESIGNACAO,DISTRITO
0,262,DF,157298,PLAT. PINGO DOCE GUARDEIRAS,Porto
1,262,DF,157567,PLAT. PINGO DOCE ALGOZ,Faro
2,262,DF,189430,PLAT. PINGO DOCE AZAMBUJA,Lisboa
3,262,DF,297283,"RAMOS & IRMÃO,LDA",PORTO
4,262,DF,313246,PLAT. LIDL TORRES NOVAS,Santarem
...,...,...,...,...,...
1932,784,Sabores,853762,AUCHAN TORRES NOVAS,Santarem
1933,784,Sabores,856542,AUCHAN VALONGO,Porto
1934,784,Sabores,856543,AUCHAN ALVERCA,NaN
1935,784,Sabores,858017,BEXEB PORTUGAL LDA,Aveiro


In [6]:

# ==========================
# 1. Identificar ficheiros com erro
# ==========================
df_erros


,FICHEIRO,ERRO
0,Factura_Andros Ambiente_Septiembre26_DEF.xlsx,Folha SALDAT027 não encontrada
1,Factura_Andros congelado_Septiembre26_DEF.xlsx,Folha SALDAT027 não encontrada
2,Factura_Andros refrigerado_Septiembre26_DEF.xlsx,Folha SALDAT027 não encontrada
3,Matutano SETEMBRO.xlsx,Folha SALDAT027 não encontrada
4,PLN Bon Natura_Septiembre.26_ok (1).xlsx,Folha SALDAT027 não encontrada


In [7]:

# ==========================
# 1. Analisar inconsistências
# ==========================
df_analise = (
    df_destinos
    .groupby(["CLIENTE", "LOCDES"], dropna=False)
    .agg(
        REGISTOS=("LOCDES", "size"),
        DESIGNACOES=("DESIGNACAO", lambda x: x.nunique(dropna=False)),
        DISTRITOS=("DISTRITO", lambda x: x.nunique(dropna=False))
    )
    .reset_index()
)

# ==========================
# 2. Identificar conflitos
# ==========================
df_conflitos = df_analise.loc[
    (df_analise["DESIGNACOES"] > 1) |
    (df_analise["DISTRITOS"] > 1)
]

df_inconsistencias = (
    df_destinos
    .merge(
        df_conflitos[["CLIENTE", "LOCDES"]],
        on=["CLIENTE", "LOCDES"],
        how="inner"
    )
    .sort_values(["CLIENTE", "LOCDES", "DESIGNACAO", "DISTRITO"])
    .reset_index(drop=True)
)

# ==========================
# 3. Resumo
# ==========================
df_resumo_conflitos = (
    df_inconsistencias
    .groupby(["CLIENTE", "NOME_CLIENTE"])
    .agg(
        REGISTOS=("LOCDES", "size"),
        LOCDES_CONFLITANTES=("LOCDES", "nunique")
    )
    .reset_index()
    .sort_values("LOCDES_CONFLITANTES", ascending=False)
)

print("Registos totais:", len(df_destinos))
print("LOCDES com conflitos:", len(df_conflitos))
print("Registos envolvidos:", len(df_inconsistencias))

display(df_resumo_conflitos)
display(df_inconsistencias)


Registos totais: 1937
LOCDES com conflitos: 78
Registos envolvidos: 297


,CLIENTE,NOME_CLIENTE,REGISTOS,LOCDES_CONFLITANTES
12,551,FLORA,118,32
3,332,Jervis,20,5
5,403,Carmonti,19,5
6,404,RTM,23,4
7,405,Lactaçores,16,4
2,282,Vidara Septiembe Facturación,14,4
13,680,Sumol Compal,19,4
8,407,Supreme Nuvifruits,11,3
14,757,Sumol MKT,14,3
10,429,Pcarnes,10,3


,CLIENTE,NOME_CLIENTE,LOCDES,DESIGNACAO,DISTRITO
0,278,TFF,754366,"AUCHAN RETAIL PORTUGAL,S.A. AZ",Lisboa
1,278,TFF,754366,AUCHAN SALVESEN,Lisboa
2,280,Simples y frescas,754366,"AUCHAN RETAIL PORTUGAL,S.A. AZ",Lisboa
3,280,Simples y frescas,754366,AUCHAN SALVESEN,Lisboa
4,282,Vidara Septiembe Facturación,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,Coimbra
...,...,...,...,...,...
292,784,Sabores,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,Coimbra
293,784,Sabores,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,MURTOSA
294,784,Sabores,579957,CMTIR,Aveiro
295,784,Sabores,579957,CMTIR,Porto


In [8]:

# ==========================
# 1. Classificar conflitos
# ==========================
df_tipos_conflito = df_conflitos.assign(
    TIPO_CONFLITO=lambda x: x.apply(
        lambda r: (
            "DESIGNACAO E DISTRITO"
            if r["DESIGNACOES"] > 1 and r["DISTRITOS"] > 1
            else "DESIGNACAO"
            if r["DESIGNACOES"] > 1
            else "DISTRITO"
        ),
        axis=1
    )
)

# ==========================
# 2. Resumo
# ==========================
df = (
    df_tipos_conflito
    .groupby("TIPO_CONFLITO")
    .size()
    .reset_index(name="TOTAL_LOCDES")
)

df


,TIPO_CONFLITO,TOTAL_LOCDES
0,DESIGNACAO,9
1,DESIGNACAO E DISTRITO,3
2,DISTRITO,66


In [9]:

# ==========================
# 1. Selecionar conflitos de distrito
# ==========================
df_distritos = df_tipos_conflito.loc[
    df_tipos_conflito["TIPO_CONFLITO"].isin([
        "DISTRITO",
        "DESIGNACAO E DISTRITO"
    ]),
    ["CLIENTE", "LOCDES"]
]

# ==========================
# 2. Consultar valores
# ==========================
df = (
    df_destinos
    .merge(df_distritos, on=["CLIENTE", "LOCDES"])
    .groupby(["CLIENTE", "NOME_CLIENTE", "LOCDES"])
    .agg(
        DESIGNACAO=("DESIGNACAO", lambda x: " | ".join(sorted(set(x.dropna())))),
        DISTRITOS=("DISTRITO", lambda x: " | ".join(sorted(set(x.dropna()))))
    )
    .reset_index()
    .sort_values(["CLIENTE", "LOCDES"])
)

df


,CLIENTE,NOME_CLIENTE,LOCDES,DESIGNACAO,DISTRITOS
0,282,Vidara Septiembe Facturación,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,Coimbra | OLIVEIRA DO BAIRRO | OURÉM
1,282,Vidara Septiembe Facturación,579957,CMTIR,Braga | Bragança | OLIVEIRA DO BAIRRO | Porto ...
2,282,Vidara Septiembe Facturación,763512,QUINTA DE JUGAIS - COMÉRCIO DE,Coimbra | OURÉM
3,282,Vidara Septiembe Facturación,763530,"CREDIN PORTUGAL, S.A.",Coimbra | Lisboa
4,332,Jervis,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,Aveiro | Coimbra | Leiria | Viseu
...,...,...,...,...,...
64,757,Sumol MKT,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,Aveiro | Coimbra | Leiria | Viseu
65,757,Sumol MKT,579957,CMTIR,Aveiro | Braga | Bragança | Porto | Viana do C...
66,757,Sumol MKT,769584,OFERTAS HORECA PRE VENDA SUL,Faro | Lisboa
67,784,Sabores,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,Aveiro | Coimbra | MURTOSA


In [11]:

# ==========================
# 1. Corrigir estrutura
# ==========================
CON.execute("""
    ALTER TABLE tarifas_mc_2026_pal
    ADD COLUMN IF NOT EXISTS NOME_CLIENTE VARCHAR
""")

CON.execute("""
    ALTER TABLE tarifas_mc_2026_pal
    ADD COLUMN IF NOT EXISTS DISTRITO_NORM VARCHAR
""")

# ==========================
# 2. Confirmar estrutura
# ==========================
df = CON.execute("""
    DESCRIBE tarifas_mc_2026_pal
""").df()

df


,column_name,column_type,null,key,default,extra
0,CLIENTE,VARCHAR,YES,None,None,None
1,DESIGNACAO,VARCHAR,YES,None,None,None
2,LOCDES,VARCHAR,YES,None,None,None
3,DISTRITO,VARCHAR,YES,None,None,None
4,PAL_1,"DECIMAL(12,4)",YES,None,None,None
5,PAL_2,"DECIMAL(12,4)",YES,None,None,None
6,PAL_3,"DECIMAL(12,4)",YES,None,None,None
7,PAL_4,"DECIMAL(12,4)",YES,None,None,None
8,PAL_5,"DECIMAL(12,4)",YES,None,None,None
9,PAL_6,"DECIMAL(12,4)",YES,None,None,None


In [12]:

# ==========================
# 1. Preparar dados
# ==========================
df_carga = df_destinos.rename(
    columns={"DISTRITO": "DISTRITO_DB"}
).copy()

df_carga["DISTRITO_NORM"] = None

# ==========================
# 2. Atualizar estrutura
# ==========================
CON.execute("BEGIN TRANSACTION")

try:
    colunas = CON.execute("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_name = 'tarifas_mc_2026_pal'
    """).df()["column_name"].tolist()

    if "DISTRITO" in colunas:
        CON.execute("""
            ALTER TABLE tarifas_mc_2026_pal
            RENAME COLUMN DISTRITO TO DISTRITO_DB
        """)

    if "DISTRITO_NORM" not in colunas:
        CON.execute("""
            ALTER TABLE tarifas_mc_2026_pal
            ADD COLUMN DISTRITO_NORM VARCHAR
        """)

    # ==========================
    # 3. Carregar destinos
    # ==========================
    CON.register("dados_destinos", df_carga)

    CON.execute("""
        INSERT INTO tarifas_mc_2026_pal (
            CLIENTE,
            NOME_CLIENTE,
            LOCDES,
            DESIGNACAO,
            DISTRITO_DB,
            DISTRITO_NORM
        )
        SELECT
            d.CLIENTE,
            d.NOME_CLIENTE,
            d.LOCDES,
            d.DESIGNACAO,
            d.DISTRITO_DB,
            d.DISTRITO_NORM
        FROM dados_destinos d
        WHERE NOT EXISTS (
            SELECT 1
            FROM tarifas_mc_2026_pal t
            WHERE t.CLIENTE IS NOT DISTINCT FROM d.CLIENTE
              AND t.LOCDES IS NOT DISTINCT FROM d.LOCDES
              AND t.DESIGNACAO IS NOT DISTINCT FROM d.DESIGNACAO
              AND t.DISTRITO_DB IS NOT DISTINCT FROM d.DISTRITO_DB
        )
    """)

    CON.execute("COMMIT")

except Exception:
    CON.execute("ROLLBACK")
    raise

finally:
    CON.unregister("dados_destinos")

# ==========================
# 4. Validar carregamento
# ==========================
df = CON.execute("""
    SELECT
        CLIENTE,
        NOME_CLIENTE,
        LOCDES,
        DESIGNACAO,
        DISTRITO_DB,
        DISTRITO_NORM
    FROM tarifas_mc_2026_pal
    ORDER BY CLIENTE, LOCDES
""").df()

df


,CLIENTE,NOME_CLIENTE,LOCDES,DESIGNACAO,DISTRITO_DB,DISTRITO_NORM
0,262,DF,157298,PLAT. PINGO DOCE GUARDEIRAS,Porto,None
1,262,DF,157567,PLAT. PINGO DOCE ALGOZ,Faro,None
2,262,DF,189430,PLAT. PINGO DOCE AZAMBUJA,Lisboa,None
3,262,DF,297283,"RAMOS & IRMÃO,LDA",PORTO,None
4,262,DF,313246,PLAT. LIDL TORRES NOVAS,Santarem,None
...,...,...,...,...,...,...
1932,784,Sabores,853762,AUCHAN TORRES NOVAS,Santarem,None
1933,784,Sabores,856542,AUCHAN VALONGO,Porto,None
1934,784,Sabores,856543,AUCHAN ALVERCA,NaN,None
1935,784,Sabores,858017,BEXEB PORTUGAL LDA,Aveiro,None


In [13]:

# ==========================
# 1. Confirmar estrutura
# ==========================
df_estrutura = CON.execute("""
    DESCRIBE tarifas_mc_2026_pal
""").df()

df_estrutura

# ==========================
# 2. Verificar registos
# ==========================
df = CON.execute("""
    SELECT COUNT(*) AS TOTAL_REGISTOS
    FROM tarifas_mc_2026_pal
""").df()

df


,TOTAL_REGISTOS
0,1937


In [14]:

# ==========================
# 1. Preparar dados
# ==========================
df_carga = (
    df_destinos
    .rename(columns={"DISTRITO": "DISTRITO_DB"})
    .drop_duplicates(subset=["CLIENTE", "LOCDES"], keep="first")
    .copy()
)

df_carga["DISTRITO_NORM"] = None

# ==========================
# 2. Carregar na DuckDB
# ==========================
CON.register("df_carga_temp", df_carga)

try:
    CON.execute("BEGIN TRANSACTION")

    CON.execute("""
        INSERT INTO tarifas_mc_2026_pal (
            CLIENTE,
            NOME_CLIENTE,
            LOCDES,
            DESIGNACAO,
            DISTRITO_DB,
            DISTRITO_NORM
        )
        SELECT
            d.CLIENTE,
            d.NOME_CLIENTE,
            d.LOCDES,
            d.DESIGNACAO,
            d.DISTRITO_DB,
            d.DISTRITO_NORM
        FROM df_carga_temp d
        WHERE NOT EXISTS (
            SELECT 1
            FROM tarifas_mc_2026_pal t
            WHERE t.CLIENTE = d.CLIENTE
              AND t.LOCDES = d.LOCDES
        )
    """)

    CON.execute("COMMIT")

except Exception:
    CON.execute("ROLLBACK")
    raise

finally:
    CON.unregister("df_carga_temp")

# ==========================
# 3. Validar carregamento
# ==========================
df = CON.execute("""
    SELECT
        COUNT(*) AS TOTAL_REGISTOS,
        COUNT(DISTINCT CLIENTE) AS CLIENTES,
        COUNT(DISTINCT (CLIENTE, LOCDES)) AS DESTINOS_UNICOS
    FROM tarifas_mc_2026_pal
""").df()

df


,TOTAL_REGISTOS,CLIENTES,DESTINOS_UNICOS
0,1937,24,1718


In [15]:

# ==========================
# 1. Ler ficheiro de tarifas
# ==========================
from pathlib import Path
import pandas as pd

FICHEIRO_TARIFAS = (
    Path.home()
    / "Documents"
    / "00.Tarifas Transporte 2026 - 20260101.xlsx"
)

excel = pd.ExcelFile(FICHEIRO_TARIFAS)

# ==========================
# 2. Identificar folhas
# ==========================
df = pd.DataFrame({
    "FOLHA": excel.sheet_names
})

df


,FOLHA
0,Notas Atualização
1,ALPRO
2,ANDROS PT
3,CARMONTI
4,CoB
5,CONFEITARIA CARLOS GONÇALVES
6,DANONE
7,DF LOGÍSTICA
8,DIA
9,ELVIRASMINDE


In [16]:

# ==========================
# 1. Ler tarifário Carmonti
# ==========================
df_carmonti = pd.read_excel(
    FICHEIRO_TARIFAS,
    sheet_name="CARMONTI",
    header=None
)

# ==========================
# 2. Visualizar estrutura
# ==========================
df = df_carmonti.iloc[:45, :22]

df


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,NaN,INFOLOG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Cliente:,403,CARMONTI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,TLD - Transporte,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,PREÇO EM €/PALETE,NaN,NaN,NaN,NaN,NaN,NaN,PREÇO EM €/PALETE,NaN,NaN,NaN,NaN,NaN
6,CENTRAIS,1/3,4/6,7/10,11/15,CARRO DEDICADO,NaN,DISTRITO,1/3,4/6,7/10,11/15,CARRO DEDICADO
7,Pingo Doce Azambuja,19.199886,17.722972,14.769143,12.083844,251.913744,NaN,Aveiro,36.30128,30.615537,24.875124,19.900099,377.145822
8,Pingo Doce Vila Conde,42.830515,39.384381,32.820318,26.852987,559.80832,NaN,Beja,45.048576,39.362834,31.982302,25.585842,484.901771
9,Pingo Doce Algóz,50.215086,46.276648,38.563873,31.55226,657.774776,NaN,Braga,50.515637,44.829894,36.424289,29.139431,552.249239


In [17]:

# ==========================
# 1. Extrair centrais
# ==========================
df_centrais = df_carmonti.iloc[7:18, 0:6].copy()

df_centrais.columns = [
    "DESTINO", "PAL_1_3", "PAL_4_6",
    "PAL_7_10", "PAL_11_15", "CARRO_DEDICADO"
]

df_centrais["TIPO_DESTINO"] = "CENTRAL"

# ==========================
# 2. Extrair distritos
# ==========================
df_distritos = df_carmonti.iloc[7:25, 7:13].copy()

df_distritos.columns = [
    "DESTINO", "PAL_1_3", "PAL_4_6",
    "PAL_7_10", "PAL_11_15", "CARRO_DEDICADO"
]

df_distritos["TIPO_DESTINO"] = "DISTRITO"

# ==========================
# 3. Consolidar
# ==========================
df_tarifas_carmonti = pd.concat(
    [df_centrais, df_distritos],
    ignore_index=True
)

df_tarifas_carmonti = df_tarifas_carmonti[
    ["TIPO_DESTINO", "DESTINO", "PAL_1_3",
     "PAL_4_6", "PAL_7_10", "PAL_11_15",
     "CARRO_DEDICADO"]
]

# ==========================
# 4. Visualizar
# ==========================
df_tarifas_carmonti


,TIPO_DESTINO,DESTINO,PAL_1_3,PAL_4_6,PAL_7_10,PAL_11_15,CARRO_DEDICADO
0,CENTRAL,Pingo Doce Azambuja,19.199886,17.722972,14.769143,12.083844,251.913744
1,CENTRAL,Pingo Doce Vila Conde,42.830515,39.384381,32.820318,26.852987,559.80832
2,CENTRAL,Pingo Doce Algóz,50.215086,46.276648,38.563873,31.55226,657.774776
3,CENTRAL,Lidl Sintra,22.153715,18.707581,15.589651,12.755169,265.908952
4,CENTRAL,Lidl Palmela,24.369086,21.415257,19.856292,17.320177,237.918536
5,CENTRAL,Lidl Marateca,24.369086,21.415257,19.856292,17.320177,237.918536
6,CENTRAL,Lidl Torres Novas,26.584457,22.153715,18.461429,15.104805,314.89218
7,CENTRAL,Lidl ST Tirso,39.186582,33.588499,27.990416,23.791854,499.628926
8,CENTRAL,Auchan Torres Novas,26.584457,22.153715,18.461429,15.104805,314.89218
9,CENTRAL,ITMP Alcanena,26.584457,22.153715,18.461429,15.104805,314.89218


In [18]:

# ==========================
# 1. Acrescentar carro completo
# ==========================
CON.execute("""
    ALTER TABLE tarifas_mc_2026_pal
    ADD COLUMN IF NOT EXISTS CARRO_COMPLETO DECIMAL(12,4)
""")

# ==========================
# 2. Confirmar estrutura
# ==========================
df = CON.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'tarifas_mc_2026_pal'
      AND column_name = 'CARRO_COMPLETO'
""").df()

df


,column_name,data_type
0,CARRO_COMPLETO,"DECIMAL(12,4)"


In [19]:

# ==========================
# 1. Localizar ficheiro de tarifas
# ==========================
from pathlib import Path

FICHEIRO_TARIFAS = (
    Path.home()
    / "Documents"
    / "00.Tarifas Transporte 2026 - 20260101.xlsx"
)

# ==========================
# 2. Confirmar ficheiro
# ==========================
print("Ficheiro:", FICHEIRO_TARIFAS)
print("Existe:", FICHEIRO_TARIFAS.is_file())


Ficheiro: C:\Users\LISARR\Documents\00.Tarifas Transporte 2026 - 20260101.xlsx
Existe: True


In [20]:

# ==========================
# 1. Ler tarifário Carmonti
# ==========================
import pandas as pd

df_excel = pd.read_excel(
    FICHEIRO_TARIFAS,
    sheet_name="CARMONTI",
    header=None
)

# ==========================
# 2. Extrair centrais
# ==========================
df_tarifas_carmonti = df_excel.iloc[7:18, 0:6].copy()

df_tarifas_carmonti.columns = [
    "DESTINO",
    "PAL_1_3",
    "PAL_4_6",
    "PAL_7_10",
    "PAL_11_15",
    "CARRO_COMPLETO"
]

df_tarifas_carmonti.insert(0, "TIPO_DESTINO", "CENTRAL")

# ==========================
# 3. Visualizar
# ==========================
df_tarifas_carmonti


,TIPO_DESTINO,DESTINO,PAL_1_3,PAL_4_6,PAL_7_10,PAL_11_15,CARRO_COMPLETO
7,CENTRAL,Pingo Doce Azambuja,19.199886,17.722972,14.769143,12.083844,251.913744
8,CENTRAL,Pingo Doce Vila Conde,42.830515,39.384381,32.820318,26.852987,559.80832
9,CENTRAL,Pingo Doce Algóz,50.215086,46.276648,38.563873,31.55226,657.774776
10,CENTRAL,Lidl Sintra,22.153715,18.707581,15.589651,12.755169,265.908952
11,CENTRAL,Lidl Palmela,24.369086,21.415257,19.856292,17.320177,237.918536
12,CENTRAL,Lidl Marateca,24.369086,21.415257,19.856292,17.320177,237.918536
13,CENTRAL,Lidl Torres Novas,26.584457,22.153715,18.461429,15.104805,314.89218
14,CENTRAL,Lidl ST Tirso,39.186582,33.588499,27.990416,23.791854,499.628926
15,CENTRAL,Auchan Torres Novas,26.584457,22.153715,18.461429,15.104805,314.89218
16,CENTRAL,ITMP Alcanena,26.584457,22.153715,18.461429,15.104805,314.89218


In [21]:

# ==========================
# 1. Bibliotecas
# ==========================
import pandas as pd
import unicodedata
import re
from difflib import SequenceMatcher

# ==========================
# 2. Normalizar designações
# ==========================
def normalizar_nome(valor):
    texto = unicodedata.normalize("NFKD", str(valor).upper())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"[^A-Z0-9]+", " ", texto)
    return " ".join(texto.split())

# ==========================
# 3. Obter centrais e destinos
# ==========================
centrais = df_tarifas_carmonti.loc[
    df_tarifas_carmonti["TIPO_DESTINO"] == "CENTRAL",
    ["DESTINO"]
].copy()

destinos = CON.execute("""
    SELECT DISTINCT LOCDES, DESIGNACAO, DISTRITO_DB
    FROM tarifas_mc_2026_pal
    WHERE CLIENTE = '403'
""").df()

centrais["NOME_NORM"] = centrais["DESTINO"].map(normalizar_nome)
destinos["NOME_NORM"] = destinos["DESIGNACAO"].map(normalizar_nome)

# ==========================
# 4. Procurar correspondências
# ==========================
resultados = []

for _, central in centrais.iterrows():
    candidatos = destinos.copy()

    candidatos["SIMILARIDADE"] = candidatos["NOME_NORM"].map(
        lambda nome: round(
            SequenceMatcher(None, central["NOME_NORM"], nome).ratio(),
            3
        )
    )

    candidatos = candidatos.sort_values(
        "SIMILARIDADE", ascending=False
    ).head(3)

    for _, candidato in candidatos.iterrows():
        resultados.append({
            "CENTRAL_TARIFARIO": central["DESTINO"],
            "LOCDES": candidato["LOCDES"],
            "DESIGNACAO_DB": candidato["DESIGNACAO"],
            "DISTRITO_DB": candidato["DISTRITO_DB"],
            "SIMILARIDADE": candidato["SIMILARIDADE"]
        })

# ==========================
# 5. Visualizar correspondências
# ==========================
df_match_carmonti = pd.DataFrame(resultados)

df_match_carmonti


,CENTRAL_TARIFARIO,LOCDES,DESIGNACAO_DB,DISTRITO_DB,SIMILARIDADE
0,Pingo Doce Azambuja,329035,PINGO-DOCE AZAMBUJA,Lisboa,1.000
1,Pingo Doce Azambuja,313255,PINGO DOCE - VILA DO CONDE,Porto,0.558
2,Pingo Doce Azambuja,313235,INDUSTRIA DE CARNES GÂNDARA,Coimbra,0.391
3,Pingo Doce Vila Conde,313255,PINGO DOCE - VILA DO CONDE,Porto,0.933
4,Pingo Doce Vila Conde,329035,PINGO-DOCE AZAMBUJA,Lisboa,0.600
5,Pingo Doce Vila Conde,313235,INDUSTRIA DE CARNES GÂNDARA,Coimbra,0.417
6,Pingo Doce Algóz,329035,PINGO-DOCE AZAMBUJA,Lisboa,0.743
7,Pingo Doce Algóz,313255,PINGO DOCE - VILA DO CONDE,Porto,0.650
8,Pingo Doce Algóz,313240,LEITÃO DOURADO,Leiria,0.467
9,Lidl Sintra,589392,PLAT. LIDL SANTO TIRSO,Porto,0.562
